# 82 — Campaign C candidate queue

S0 失敗後に **次候補を一覧・archive・materialize** する。

- GPU M2 は起動しない
- production M6 は開始しない
- `NO_SELECTION` 済み候補は `ARCHIVE.json` を書いてスキップする


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm7_candidate_queue.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m7_candidate_queue.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.m7_candidate_queue import (
    search_root_for, list_queue_rows, next_actionable_candidate,
    materialize_top_k, package_root_for,
)
from src.m7_status import M7_RUN_ID_CAMPAIGN_C

M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', M7_RUN_ID_CAMPAIGN_C)
SEARCH_ROOT = search_root_for(PERSIST_ROOT, M7C_RUN_ID)
print('SEARCH_ROOT', SEARCH_ROOT)

ROWS = list_queue_rows(SEARCH_ROOT, persistent_root=PERSIST_ROOT)
print(f'queue rows: {len(ROWS)}')
print('| cand | j2 | est_q | mat | arch | m2 | s0 |')
print('|---|---:|---:|:---:|:---:|:---:|:---|')
for row in ROWS[:20]:
    print(
        f"| {row['candidate_id']} | {row['j2_max']} | {row['estimated_q']:.4g} | "
        f"{'Y' if row['materialized'] else '-'} | "
        f"{row.get('archive_reason') or ('Y' if row['archived'] else '-')} | "
        f"{'Y' if row['m2_ready'] else '-'} | "
        f"{row.get('s0_selection_status') or '-'} |"
    )


## Backfill archive for CAND-000005 (known NO_SELECTION sweep)


In [ ]:
from src.m7_archive import archive_from_sweep, is_archived, read_archive

CAND_000005 = 'CAND-000005-ac85c5e9ba38'
PKG_000005 = package_root_for(SEARCH_ROOT, CAND_000005)
SWEEP_000005 = PKG_000005 / 'rank_sweep' / 'SWEEP-20260720T120952Z-1bc3b808bd46'
if is_archived(PKG_000005):
    print('already archived', read_archive(PKG_000005))
elif SWEEP_000005.is_dir():
    arch = archive_from_sweep(
        PKG_000005,
        SWEEP_000005,
        selection_status='NO_SELECTION',
        selection_reasons=[
            'rank=16: q_optimistic>=1',
            'rank=24..96: mid-cluster',
        ],
        overwrite=False,
    )
    print(json.dumps(arch, indent=2, ensure_ascii=False))
else:
    print('Sweep dir missing; set SWEEP path manually or skip.')
    print('expected', SWEEP_000005)


## Materialize top-K staged candidates (no GPU) + show next


In [ ]:
K = int(os.environ.get('VALIDATED_RG_MATERIALIZE_TOP_K', '3'))
created = materialize_top_k(SEARCH_ROOT, persistent_root=PERSIST_ROOT, k=K)
print('materialized/ensured', len(created))
for m in created:
    print('-', m.get('candidate_id'), m.get('package_root'))

NXT = next_actionable_candidate(SEARCH_ROOT, persistent_root=PERSIST_ROOT)
if NXT is None:
    print('No actionable candidate left (all archived or gated).')
else:
    print('NEXT candidate:', NXT['candidate_id'])
    print('j2_max', NXT['j2_max'], 'est_q', NXT['estimated_q'])
    print('m2_ready', NXT['m2_ready'], 'materialized', NXT['materialized'])
    pkg = package_root_for(SEARCH_ROOT, NXT['candidate_id'])
    print('export VALIDATED_RG_STAGED_CANDIDATE=' + NXT['candidate_id'])
    print('export VALIDATED_RG_STAGED_PACKAGE=' + str(pkg))
    if not NXT['m2_ready']:
        print('NEXT STEP: notebook 73_m7_staged_s3_lineage.ipynb with that env')
    else:
        print('NEXT STEP: notebook 83_s0_rank_sweep_series.ipynb')


## Pre-M2 promotion (no S0 dependency) + resolve shared binding


In [ ]:
from src.m7_promotion import evaluate_promotion, ranking_snapshot_hashes
from src.m2_shared_registry import resolve_m2_binding
from src.m7_candidate_queue import list_queue_rows, next_actionable_candidate, package_root_for

ROWS = list_queue_rows(SEARCH_ROOT, persistent_root=PERSIST_ROOT)
snap = ranking_snapshot_hashes([r['ranking_row'] for r in ROWS])
live = [r for r in ROWS if r.get('staged_executable') or r.get('instant_executable')]
for idx, row in enumerate(live[:5]):
    pkg = package_root_for(SEARCH_ROOT, row['candidate_id'])
    if not pkg.is_dir():
        continue
    evaluate_promotion(
        package_root=pkg,
        j2_max=row['j2_max'],
        estimated_q=row['estimated_q'],
        rank_among_executable=idx,
        top_k=3,
        campaign_run_id=M7C_RUN_ID,
        ranking_snapshot_sha256=snap['ranking_snapshot_sha256'],
        candidate_set_sha256=snap['candidate_set_sha256'],
    )
    try:
        binding = resolve_m2_binding(
            persistent_root=PERSIST_ROOT,
            project_root=PROJECT_ROOT,
            package_root=pkg,
            j2_max=row['j2_max'],
        )
        print(row['candidate_id'], 'state', binding.get('state'), 'mode', binding.get('mode'))
    except Exception as exc:
        print(row['candidate_id'], 'resolve failed', exc)

NXT = next_actionable_candidate(SEARCH_ROOT, persistent_root=PERSIST_ROOT)
print('NEXT', None if NXT is None else NXT['candidate_id'])
